### Setup & Imports

Load required libraries (pandas, numpy, pathlib) and define file paths to sample data.
This prepares the environment for working with the Excel files.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
SAMPLES_DIR = Path("samples")
MASTER_FILE = SAMPLES_DIR / "Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx"
BSNY_CONCUR_FILE = SAMPLES_DIR / "BSNY - SAP & Concur Repoort May 2025 - June 2026.xlsx"
SANCAP_CONCUR_FILE = SAMPLES_DIR / "SanCap - Expense Report USA May 2025 - JUNE 2026.xlsx"


# ------------------------------------------------------------------------------------------
print("✓ Dependencies loaded")
print(f"✓ Sample files directory: {SAMPLES_DIR}")
print(f"✓ Files ready to process")

✓ Dependencies loaded
✓ Sample files directory: samples
✓ Files ready to process


### Prepare source files

In [15]:
# STEP 1: Confirm the source reports exist
print("=" * 80)
print("STEP 1: Confirm Source Reports")
print("=" * 80)

# Check BSNY file
print(f"\n1. BSNY - SAP & Concur Report:")
print(f"   File exists: {BSNY_CONCUR_FILE.exists()}")
if BSNY_CONCUR_FILE.exists():
    bsny_wb = pd.ExcelFile(BSNY_CONCUR_FILE)
    print(f"   Sheets available: {bsny_wb.sheet_names}")
    print(f"   → Has 'Concur' tab: {'Concur' in bsny_wb.sheet_names}")

# Check SANCAP file
print(f"\n2. SanCap - Expense Report USA:")
print(f"   File exists: {SANCAP_CONCUR_FILE.exists()}")
if SANCAP_CONCUR_FILE.exists():
    sancap_wb = pd.ExcelFile(SANCAP_CONCUR_FILE)
    print(f"   Sheets available: {sancap_wb.sheet_names}")

print("\n✓ Source reports confirmed")

STEP 1: Confirm Source Reports

1. BSNY - SAP & Concur Report:
   File exists: True
   Sheets available: ['Concur', 'SAP']
   → Has 'Concur' tab: True

2. SanCap - Expense Report USA:
   File exists: True
   Sheets available: ['page']

✓ Source reports confirmed


### Load Prior-Month Combined Master File

In [16]:
print("\n" + "=" * 80)
print("STEP 2: Load Prior-Month Combined Master File")
print("=" * 80)

print(f"\nLoading: {MASTER_FILE.name}")
master_wb = pd.ExcelFile(MASTER_FILE)
print(f"Sheets in master file: {master_wb.sheet_names}")

# We'll need to reference this for comparison
# Let's load the prior Concur data from the master file
print("\n✓ Prior-month master file loaded")
print("  (Will use this for sanity check in Step 4)")


STEP 2: Load Prior-Month Combined Master File

Loading: Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx
Sheets in master file: ['Coding', 'Combined Summary', 'Concur Report', 'SAP Report', 'Uber Report', 'Concur Analysis - PIVOTS', 'SAP Invoices Analysis - PIVOTS', 'Uber Invoices Analysis - PIVOTS', 'All CCs', 'HR HC Combined Jun', 'HR Headcount', 'In Scope CCs']

✓ Prior-month master file loaded
  (Will use this for sanity check in Step 4)


### Apply 2026 Reporting-Period Filter

In [17]:
# STEP 3: Find Year column (should be BA) and filter to 2026
print("\n" + "=" * 80)
print("STEP 3: Apply 2026 Reporting Period Filter")
print("=" * 80)

# Load BSNY Concur tab
print("\n1. BSNY - Concur Tab:")
bsny_concur = pd.read_excel(BSNY_CONCUR_FILE, sheet_name="Concur")
print(f"   Loaded: {bsny_concur.shape[0]} rows × {bsny_concur.shape[1]} columns")

# Find Year column
year_col = None
for col in bsny_concur.columns:
    if 'Year' in str(col) or 'year' in str(col).lower():
        year_col = col
        print(f"   → Found Year column: '{col}'")
        break

if year_col is None:
    # Check if it's column BA (column 53)
    if bsny_concur.shape[1] >= 53:
        year_col = bsny_concur.columns[52]  # BA is column 53 (0-indexed = 52)
        print(f"   → Using column 53 (BA): '{year_col}'")

# Filter to 2026
bsny_concur_2026 = bsny_concur[bsny_concur[year_col] == 2026].copy()
print(f"   → Filtered to Year=2026: {bsny_concur_2026.shape[0]} rows")

# Load SanCap - only 1 sheet, so load directly
print("\n2. SanCap - Expense Report:")
sancap_concur = pd.read_excel(SANCAP_CONCUR_FILE)
print(f"   Loaded: {sancap_concur.shape[0]} rows × {sancap_concur.shape[1]} columns")

sancap_concur_2026 = sancap_concur[sancap_concur[year_col] == 2026].copy()
print(f"   → Filtered to Year=2026: {sancap_concur_2026.shape[0]} rows")

print("\n✓ 2026 data filtered for both entities")

# Display sample data: first 5 and last 5 rows
print("\n" + "=" * 80)
print("SAMPLE DATA - First 5 rows")
print("=" * 80)

# Get first 5 columns + year column
first_cols = list(bsny_concur_2026.columns[:5]) + [year_col]
print("\nBSNY Concur 2026 - FIRST 5 ROWS:")
print(bsny_concur_2026[first_cols].head(5).to_string())

print("\n\nSanCap Concur 2026 - FIRST 5 ROWS:")
print(sancap_concur_2026[first_cols].head(5).to_string())

print("\n" + "=" * 80)
print("SAMPLE DATA - Last 5 rows")
print("=" * 80)

print("\nBSNY Concur 2026 - LAST 5 ROWS:")
print(bsny_concur_2026[first_cols].tail(5).to_string())

print("\n\nSanCap Concur 2026 - LAST 5 ROWS:")
print(sancap_concur_2026[first_cols].tail(5).to_string())


STEP 3: Apply 2026 Reporting Period Filter

1. BSNY - Concur Tab:
   Loaded: 34118 rows × 72 columns
   → Found Year column: 'Year'
   → Filtered to Year=2026: 15377 rows

2. SanCap - Expense Report:
   Loaded: 68915 rows × 72 columns
   → Filtered to Year=2026: 33177 rows

✓ 2026 data filtered for both entities

SAMPLE DATA - First 5 rows

BSNY Concur 2026 - FIRST 5 ROWS:
   Employee First Name Employee Last Name            Employee Login ID  Employee ID       Report Name  Year
13              Gareth             Davies  n100167@SCIBUS.santander.us       100167             April  2026
14              Gareth             Davies  n100167@SCIBUS.santander.us       100167             April  2026
42              Gareth             Davies  n100167@SCIBUS.santander.us       100167     December 2025  2026
43              Gareth             Davies  n100167@SCIBUS.santander.us       100167     December 2025  2026
91              Gareth             Davies  n100167@SCIBUS.santander.us       100167

### Sanity Check: Compare Totals to Prior Month


In [13]:
"""
# STEP 4: Sanity Check - Compare Totals to Prior Month (IN-SCOPE ONLY)
print("\n" + "=" * 80)
print("STEP 4: Sanity Check - Compare Totals to Prior Month (IN-SCOPE ONLY)")
print("=" * 80)

# Find Expense Amount column (should be exactly "Expense Amount (reimbursement currency)")
expense_col = None
for col in bsny_concur.columns:
    if col == "Expense Amount (reimbursement currency)":
        expense_col = col
        print(f"   Found Expense Amount column: '{col}'")
        break

if expense_col is None:
    print(f"   ✗ ERROR: Column 'Expense Amount (reimbursement currency)' not found")
    print(f"   Available columns containing 'amount':")
    for col in bsny_concur.columns:
        if 'amount' in str(col).lower():
            print(f"      - {col}")
else:
    try:
        # Load PRIOR MONTH from master file
        print(f"\n📄 Loading prior-month data from 'Concur Report' sheet...")
        master_concur = pd.read_excel(MASTER_FILE, sheet_name="Concur Report")
        print(f"   Total rows in prior month: {len(master_concur)}")
        
        # Filter PRIOR MONTH to only IN-SCOPE rows using "In Scope" column
        if "In Scope" in master_concur.columns:
            master_in_scope = master_concur[master_concur["In Scope"] == "Yes"]
            prior_total_in_scope = master_in_scope[expense_col].sum()
            
            print(f"\n📊 PRIOR MONTH (IN-SCOPE ONLY):")
            print(f"   Total rows in prior month:       {len(master_concur)}")
            print(f"   Rows marked as 'In Scope: Yes':  {len(master_in_scope)}")
            print(f"   Prior Total (In-Scope):          ${prior_total_in_scope:>15,.2f}")
        else:
            print(f"   ✗ ERROR: 'In Scope' column not found in Concur Report sheet")
            prior_total_in_scope = 0
            master_in_scope = pd.DataFrame()
        
        # CURRENT MONTH: Sum ALL 2026 data (in-scope marking happens in Step 10)
        bsny_current = bsny_concur_2026[expense_col].sum()
        sancap_current = sancap_concur_2026[expense_col].sum()
        combined_current = bsny_current + sancap_current
        
        print(f"\n📊 CURRENT MONTH (2026 - ALL DATA):")
        print(f"   BSNY Concur:     ${bsny_current:>15,.2f}")
        print(f"   SanCap Concur:   ${sancap_current:>15,.2f}")
        print(f"   ─────────────────────────────────")
        print(f"   Combined Total:  ${combined_current:>15,.2f}")
        print(f"\n   Note: New data will be marked as 'In Scope: Yes/No' in Step 10")
        print(f"         (After formulas in 'CC Expense is Mapped to' are applied)")
        
        # SANITY CHECK: Compare current vs prior in-scope
        print(f"\n" + "=" * 80)
        print(f"✓ SANITY CHECK COMPARISON:")
        print(f"=" * 80)
        print(f"   Current Month (all 2026):   ${combined_current:>15,.2f}")
        print(f"   Prior Month (in-scope only): ${prior_total_in_scope:>15,.2f}")
        print(f"   Difference:                  ${combined_current - prior_total_in_scope:>15,.2f}")
        
        if combined_current >= prior_total_in_scope:
            print(f"\n✅ PASS: Current >= Prior (In-Scope)")
            print(f"   YTD totals are increasing correctly ✓")
            print(f"   Data ready for refresh to Step 6 ✓")
        else:
            print(f"\n❌ FAIL: Current < Prior (In-Scope)!")
            print(f"\n   ⚠️  ERROR: The total DECREASED! This should not happen for year-to-date data.")
            print(f"   Action: STOP - Do not continue with refresh!")
            print(f"   Reason: Check if data was correctly filtered to 2026")
            
    except Exception as e:
        print(f"   ✗ ERROR during sanity check: {e}")
        print(f"   Traceback: {type(e).__name__}")
"""


STEP 4: Sanity Check - Compare Totals to Prior Month (IN-SCOPE ONLY)
   Found Expense Amount column: 'Expense Amount (reimbursement currency)'

📄 Loading prior-month data from 'Concur Report' sheet...
   Total rows in prior month: 48554

📊 PRIOR MONTH (IN-SCOPE ONLY):
   Total rows in prior month:       48554
   Rows marked as 'In Scope: Yes':  10585
   Prior Total (In-Scope):          $   2,364,474.49

📊 CURRENT MONTH (2026 - ALL DATA):
   BSNY Concur:     $   3,022,379.23
   SanCap Concur:   $   7,783,763.45
   ─────────────────────────────────
   Combined Total:  $  10,806,142.68

   Note: New data will be marked as 'In Scope: Yes/No' in Step 10
         (After formulas in 'CC Expense is Mapped to' are applied)

✓ SANITY CHECK COMPARISON:
   Current Month (all 2026):   $  10,806,142.68
   Prior Month (in-scope only): $   2,364,474.49
   Difference:                  $   8,441,668.19

✅ PASS: Current >= Prior (In-Scope)
   YTD totals are increasing correctly ✓
   Data ready for refre

### Master File Column Structure Analysis

In [22]:
# STEP 5: Master File Column Structure Analysis
print("=" * 100)
print("STEP 5: Master File Column Structure Analysis")
print("=" * 100)

# Load Master file
master_df = pd.read_excel(MASTER_FILE, sheet_name="Concur Report")

print(f"\n📊 Master File - 'Concur Report' Sheet")
print(f"   Total rows: {len(master_df)}")
print(f"   Total columns: {len(master_df.columns)}")

# Helper columns to identify
helper_columns = ["First & Last Name", "CC Expense is Mapped to", "LOB", "In Scope", "Error"]

# Create detailed column mapping
column_info = []
for idx, col_name in enumerate(master_df.columns):
    # Convert index to column letter
    col_letter = ""
    n = idx
    while n >= 0:
        col_letter = chr(65 + (n % 26)) + col_letter
        n = n // 26 - 1
    
    is_helper = "YES ✓ HELPER" if col_name in helper_columns else "No"
    data_type = str(master_df[col_name].dtype)
    
    column_info.append({
        'Position': col_letter,
        'Index': idx,
        'Column Name': col_name,
        'Type': data_type,
        'Helper?': is_helper
    })

# Display key columns
print("\nKey Columns:")
key_cols_to_show = ["Year", "First Name", "Last Name", "First & Last Name", "CC Expense is Mapped to", 
                     "LOB", "In Scope", "Error"]
for col_info_dict in column_info:
    if col_info_dict['Column Name'] in key_cols_to_show:
        print(f"   {col_info_dict['Position']:>3} | Idx {col_info_dict['Index']:>3} | {col_info_dict['Column Name']:<30} | {col_info_dict['Type']:<10} | {col_info_dict['Helper?']}")

# Identify helper columns
print("\n" + "─" * 100)
print("HELPER COLUMNS (These have formulas, NOT replaced with source data):")
print("─" * 100)
for col_name in helper_columns:
    if col_name in master_df.columns:
        col_idx = list(master_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ {col_letter:>3} | {col_name:<30}")
    else:
        print(f"   ✗ NOT FOUND: {col_name}")

# Sample data
print("\n" + "─" * 100)
print("SAMPLE DATA - First 3 rows:")
print("─" * 100)
print(master_df.head(3).to_string())

# Row count by In Scope status
print("\n" + "─" * 100)
print("ROW COUNT SUMMARY:")
print("─" * 100)
print(f"Total rows in Master: {len(master_df)}")
if "In Scope" in master_df.columns:
    scope_counts = master_df["In Scope"].value_counts()
    print(scope_counts)

print("\n✓ Step 5 Complete")

STEP 5: Master File Column Structure Analysis

📊 Master File - 'Concur Report' Sheet
   Total rows: 48554
   Total columns: 77

Key Columns:
     C | Idx   2 | First & Last Name              | str        | YES ✓ HELPER
    BB | Idx  53 | Year                           | int64      | No
    BV | Idx  73 | CC Expense is Mapped to        | str        | YES ✓ HELPER
    BW | Idx  74 | LOB                            | str        | YES ✓ HELPER
    BX | Idx  75 | In Scope                       | str        | YES ✓ HELPER
    BY | Idx  76 | Error                          | str        | YES ✓ HELPER

────────────────────────────────────────────────────────────────────────────────────────────────────
HELPER COLUMNS (These have formulas, NOT replaced with source data):
────────────────────────────────────────────────────────────────────────────────────────────────────
   ✓   C | First & Last Name             
   ✓  BV | CC Expense is Mapped to       
   ✓  BW | LOB                           
   

### BSNY Source File Column Analysis

In [23]:
print("\n" + "=" * 100)
print("STEP 6: BSNY Source File Column Analysis")
print("=" * 100)

# Load BSNY file
bsny_df = pd.read_excel(BSNY_CONCUR_FILE, sheet_name="Concur")

print(f"\n📊 BSNY File - 'Concur' Sheet")
print(f"   Total rows: {len(bsny_df)}")
print(f"   Total columns: {len(bsny_df.columns)}")

# Find Year column
year_col_bsny = None
for col in bsny_df.columns:
    if 'Year' in str(col) or col.lower() == 'year':
        year_col_bsny = col
        break

if year_col_bsny is None and len(bsny_df.columns) > 52:
    year_col_bsny = bsny_df.columns[52]

print(f"\n   Year column found: '{year_col_bsny}'")

# Find Custom columns
print("\n" + "─" * 100)
print("CUSTOM COLUMNS (Mapping targets):")
print("─" * 100)

for col_name in bsny_df.columns:
    if 'Custom 41' in str(col_name):
        col_idx = list(bsny_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 41: '{col_name}' at position {col_letter}")
    elif 'Custom 42' in str(col_name):
        col_idx = list(bsny_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 42: '{col_name}' at position {col_letter}")
    elif 'Custom 43' in str(col_name):
        col_idx = list(bsny_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 43: '{col_name}' at position {col_letter}")

# Row breakdown by Year
print("\n" + "─" * 100)
print("ROW BREAKDOWN BY YEAR:")
print("─" * 100)
year_counts = bsny_df[year_col_bsny].value_counts().sort_index(ascending=False)
print(year_counts)

# Sum expense amounts by Year
if "Expense Amount (reimbursement currency)" in bsny_df.columns:
    print("\n" + "─" * 100)
    print("EXPENSE AMOUNT TOTALS BY YEAR:")
    print("─" * 100)
    expense_by_year = bsny_df.groupby(year_col_bsny)["Expense Amount (reimbursement currency)"].sum().sort_index(ascending=False)
    for year, amount in expense_by_year.items():
        print(f"   {year}: ${amount:>15,.2f}")

# Filter to 2026 for later use
bsny_2026_raw = bsny_df[bsny_df[year_col_bsny] == 2026].copy()
print(f"\n📌 BSNY 2026 Data: {len(bsny_2026_raw)} rows (to be inserted)")

print("\n✓ Step 6 Complete")


STEP 6: BSNY Source File Column Analysis

📊 BSNY File - 'Concur' Sheet
   Total rows: 34118
   Total columns: 72

   Year column found: 'Year'

────────────────────────────────────────────────────────────────────────────────────────────────────
CUSTOM COLUMNS (Mapping targets):
────────────────────────────────────────────────────────────────────────────────────────────────────
   ✓ Custom 41: 'Custom 41 - Name' at position BR
   ✓ Custom 42: 'Custom 42 - Name' at position BS
   ✓ Custom 43: 'Custom 43 - Name' at position BT

────────────────────────────────────────────────────────────────────────────────────────────────────
ROW BREAKDOWN BY YEAR:
────────────────────────────────────────────────────────────────────────────────────────────────────
Year
2026    15377
2025    18741
Name: count, dtype: int64

────────────────────────────────────────────────────────────────────────────────────────────────────
EXPENSE AMOUNT TOTALS BY YEAR:
───────────────────────────────────────────────────

### SanCap Source File Column Analysis

In [24]:
print("\n" + "=" * 100)
print("STEP 7: SanCap Source File Column Analysis")
print("=" * 100)

# Load SanCap file (single sheet)
sancap_df = pd.read_excel(SANCAP_CONCUR_FILE)

print(f"\n📊 SanCap File (single sheet)")
print(f"   Total rows: {len(sancap_df)}")
print(f"   Total columns: {len(sancap_df.columns)}")

# Find Year column
year_col_sancap = None
for col in sancap_df.columns:
    if 'Year' in str(col) or col.lower() == 'year':
        year_col_sancap = col
        break

if year_col_sancap is None and len(sancap_df.columns) > 52:
    year_col_sancap = sancap_df.columns[52]

print(f"\n   Year column found: '{year_col_sancap}'")

# Find Custom columns
print("\n" + "─" * 100)
print("CUSTOM COLUMNS (Mapping targets):")
print("─" * 100)

for col_name in sancap_df.columns:
    if 'Custom 41' in str(col_name):
        col_idx = list(sancap_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 41: '{col_name}' at position {col_letter}")
    elif 'Custom 42' in str(col_name):
        col_idx = list(sancap_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 42: '{col_name}' at position {col_letter}")
    elif 'Custom 43' in str(col_name):
        col_idx = list(sancap_df.columns).index(col_name)
        col_letter = ""
        n = col_idx
        while n >= 0:
            col_letter = chr(65 + (n % 26)) + col_letter
            n = n // 26 - 1
        print(f"   ✓ Custom 43: '{col_name}' at position {col_letter}")

# Row breakdown by Year
print("\n" + "─" * 100)
print("ROW BREAKDOWN BY YEAR:")
print("─" * 100)
year_counts_sancap = sancap_df[year_col_sancap].value_counts().sort_index(ascending=False)
print(year_counts_sancap)

# Sum expense amounts by Year
if "Expense Amount (reimbursement currency)" in sancap_df.columns:
    print("\n" + "─" * 100)
    print("EXPENSE AMOUNT TOTALS BY YEAR:")
    print("─" * 100)
    expense_by_year_sancap = sancap_df.groupby(year_col_sancap)["Expense Amount (reimbursement currency)"].sum().sort_index(ascending=False)
    for year, amount in expense_by_year_sancap.items():
        print(f"   {year}: ${amount:>15,.2f}")

# Filter to 2026 for later use
sancap_2026_raw = sancap_df[sancap_df[year_col_sancap] == 2026].copy()
print(f"\n📌 SanCap 2026 Data: {len(sancap_2026_raw)} rows (to be inserted)")

print("\n✓ Step 7 Complete")


STEP 7: SanCap Source File Column Analysis

📊 SanCap File (single sheet)
   Total rows: 68915
   Total columns: 72

   Year column found: 'Year'

────────────────────────────────────────────────────────────────────────────────────────────────────
CUSTOM COLUMNS (Mapping targets):
────────────────────────────────────────────────────────────────────────────────────────────────────
   ✓ Custom 41: 'Custom 41 - Name' at position BR
   ✓ Custom 42: 'Custom 42 - Name' at position BS
   ✓ Custom 43: 'Custom 43 - Name' at position BT

────────────────────────────────────────────────────────────────────────────────────────────────────
ROW BREAKDOWN BY YEAR:
────────────────────────────────────────────────────────────────────────────────────────────────────
Year
2026    33177
2025    35738
Name: count, dtype: int64

────────────────────────────────────────────────────────────────────────────────────────────────────
EXPENSE AMOUNT TOTALS BY YEAR:
─────────────────────────────────────────────────

### Column Mapping Validation

In [21]:
print("\n" + "=" * 100)
print("STEP 8: Column Mapping Validation")
print("=" * 100)

# Define the 3 required mappings
mappings = [
    {'source': 'Custom 41 - Name', 'target': 'Client Name'},
    {'source': 'Custom 42 - Name', 'target': 'Project Name'},
    {'source': 'Custom 43 - Name', 'target': 'Epense Name'}
]

print("\n" + "─" * 100)
print("MAPPING VALIDATION TABLE:")
print("─" * 100)

all_mappings_valid = True

for mapping in mappings:
    source_col = mapping['source']
    target_col = mapping['target']
    
    # Check source in BSNY
    bsny_source_exists = source_col in bsny_df.columns
    # Check source in SanCap
    sancap_source_exists = source_col in sancap_df.columns
    # Check target in Master
    master_target_exists = target_col in master_df.columns
    
    # Status
    status = "✓ VALID" if (bsny_source_exists and sancap_source_exists and master_target_exists) else "✗ ERROR"
    if status == "✗ ERROR":
        all_mappings_valid = False
    
    print(f"\n{status}: {source_col} → {target_col}")
    print(f"   BSNY Source: {'✓ Found' if bsny_source_exists else '✗ NOT FOUND'}")
    print(f"   SanCap Source: {'✓ Found' if sancap_source_exists else '✗ NOT FOUND'}")
    print(f"   Master Target: {'✓ Found' if master_target_exists else '✗ NOT FOUND'}")

print("\n" + "─" * 100)
if all_mappings_valid:
    print("✓ ALL MAPPINGS VALID - Ready for Step 9")
else:
    print("✗ SOME MAPPINGS INVALID - Check errors above")
print("─" * 100)

print("\n✓ Step 8 Complete")


STEP 8: Column Mapping Validation

────────────────────────────────────────────────────────────────────────────────────────────────────
MAPPING VALIDATION TABLE:
────────────────────────────────────────────────────────────────────────────────────────────────────


NameError: name 'master_df' is not defined

### Row Count & Totals Summary

In [10]:
print("\n" + "=" * 100)
print("STEP 9: Row Count & Totals Summary")
print("=" * 100)

print("\n" + "─" * 100)
print("SUMMARY TABLE:")
print("─" * 100)

# Get metrics
master_total_rows = len(master_df)
bsny_2026_rows = len(bsny_2026_raw)
sancap_2026_rows = len(sancap_2026_raw)
combined_2026_rows = bsny_2026_rows + sancap_2026_rows

# Calculate expected final row count
expected_final_rows = 1 + combined_2026_rows  # 1 for header

# Expense totals
bsny_2026_expense = bsny_2026_raw["Expense Amount (reimbursement currency)"].sum() if "Expense Amount (reimbursement currency)" in bsny_2026_raw.columns else 0
sancap_2026_expense = sancap_2026_raw["Expense Amount (reimbursement currency)"].sum() if "Expense Amount (reimbursement currency)" in sancap_2026_raw.columns else 0
combined_2026_expense = bsny_2026_expense + sancap_2026_expense

# Print summary
summary_data = {
    'Metric': [
        'Master Current Rows (total)',
        'BSNY 2026 Rows (new)',
        'SanCap 2026 Rows (new)',
        'Combined 2026 Rows',
        '',
        'Expected Final Rows (after refresh)',
        '',
        'BSNY 2026 Expense Total',
        'SanCap 2026 Expense Total',
        'Combined 2026 Expense Total'
    ],
    'Value': [
        f"{master_total_rows:,}",
        f"{bsny_2026_rows:,}",
        f"{sancap_2026_rows:,}",
        f"{combined_2026_rows:,}",
        '',
        f"{expected_final_rows:,}",
        '',
        f"${bsny_2026_expense:,.2f}",
        f"${sancap_2026_expense:,.2f}",
        f"${combined_2026_expense:,.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("\n" + "─" * 100)
print("VALIDATION CHECKS:")
print("─" * 100)
print(f"✓ Master file has data: {master_total_rows > 0}")
print(f"✓ BSNY 2026 data exists: {bsny_2026_rows > 0}")
print(f"✓ SanCap 2026 data exists: {sancap_2026_rows > 0}")
print(f"✓ Combined data ready: {combined_2026_rows > 0}")
print(f"✓ All mappings valid: {all_mappings_valid}")

print("\n✓ Step 9 Complete - Ready for Step 10!")
print("\n📌 Next: Step 10 will create outputs folder, copy Master, and insert new data with formulas")


STEP 9: Row Count & Totals Summary

────────────────────────────────────────────────────────────────────────────────────────────────────
SUMMARY TABLE:
────────────────────────────────────────────────────────────────────────────────────────────────────
                             Metric          Value
        Master Current Rows (total)         48,554
               BSNY 2026 Rows (new)         15,377
             SanCap 2026 Rows (new)         33,177
                 Combined 2026 Rows         48,554
                                                  
Expected Final Rows (after refresh)         48,555
                                                  
            BSNY 2026 Expense Total  $3,022,379.23
          SanCap 2026 Expense Total  $7,783,763.45
        Combined 2026 Expense Total $10,806,142.68

────────────────────────────────────────────────────────────────────────────────────────────────────
VALIDATION CHECKS:
────────────────────────────────────────────────────────────────

## Data Refresh (Step 10 = 3.1.2):

### Create Outputs Folder & Copy Master File

In [3]:
# STEP 10A: COM refresh preflight (this cell does not modify any workbook)
from pathlib import Path
import shutil
import win32com.client as win32

OUTPUT_DIR = Path("outputs")
OUTPUT_MASTER_FILE = OUTPUT_DIR / f"{MASTER_FILE.stem}_paste_concur{MASTER_FILE.suffix}"

required_files = [MASTER_FILE, BSNY_CONCUR_FILE, SANCAP_CONCUR_FILE]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Required workbook(s) not found:\n" + "\n".join(missing_files))

OUTPUT_DIR.mkdir(exist_ok=True)
print("STEP 10A: COM Refresh Preflight")
print(f"Master input: {MASTER_FILE.name}")
print(f"Output file:  {OUTPUT_MASTER_FILE}")
print("The existing output file will be overwritten after Step 10B validation succeeds.")

STEP 10A: COM Refresh Preflight
Master input: Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx
Output file:  outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED_paste_concur.xlsx
The existing output file will be overwritten after Step 10B validation succeeds.


### Refresh Concur Report with Excel COM

In [4]:
# STEP 10B: Create the refreshed master workbook using Excel COM
# Run Step 10A first. This code only writes to OUTPUT_MASTER_FILE, never MASTER_FILE.

MASTER_SHEET_NAME = "Concur Report"
BSNY_SHEET_NAME = "Concur"
HELPER_HEADERS = [
    "First & Last Name",
    "CC Expense is Mapped to",
    "LOB",
    "In Scope",
    "Error",
]
SOURCE_TO_MASTER = {
    "Custom 41 - Name": "Client Name",
    "Custom 42 - Name": "Project Name",
    "Custom 43 - Name": "Epense Name",
}
YEAR_HEADER = "Year"
EXPENSE_HEADER = "Expense Amount (reimbursement currency)"
XL_UP = -4162
XL_TO_LEFT = -4159
XL_CALCULATION_MANUAL = -4135
XL_CALCULATION_AUTOMATIC = -4105
XL_PASTE_VALUES = -4163
XL_SHEET_VERY_HIDDEN = 2


def normalise_header(value):
    """Return a consistent Excel header value, rejecting blank headers."""
    if value is None:
        return ""
    return str(value).strip()


def read_headers(worksheet):
    """Read row 1 and return every column position for each non-blank header."""
    last_column = worksheet.Cells(1, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    raw_headers = worksheet.Range(worksheet.Cells(1, 1), worksheet.Cells(1, last_column)).Value2
    header_values = list(raw_headers[0]) if isinstance(raw_headers, tuple) else [raw_headers]
    headers = {}

    for column_number, raw_header in enumerate(header_values, start=1):
        header = normalise_header(raw_header)
        if not header:
            continue
        headers.setdefault(header, []).append(column_number)
    return headers


def unique_header_column(headers, header, worksheet):
    """Return one required column, rejecting missing or repeated structural headers."""
    columns = headers.get(header, [])
    if len(columns) != 1:
        description = "missing" if not columns else f"repeated in columns {columns}"
        raise ValueError(f"Required header '{header}' is {description} in '{worksheet.Name}'.")
    return columns[0]


def last_used_row(worksheet):
    """Find the last row containing a value or formula."""
    return worksheet.Cells(worksheet.Rows.Count, 1).End(XL_UP).Row


def column_values(worksheet, column_number, last_row):
    """Return values from Excel rows 2:last_row as a Python list."""
    if last_row < 2:
        return []
    raw_values = worksheet.Range(worksheet.Cells(2, column_number), worksheet.Cells(last_row, column_number)).Value2
    if last_row == 2:
        return [raw_values]
    return [row[0] for row in raw_values]


def is_reporting_year(value, year=2026):
    """Support Excel numeric and text representations of the reporting year."""
    try:
        return float(value) == float(year)
    except (TypeError, ValueError):
        return str(value).strip() == str(year)


def source_rows_for_year(worksheet, headers):
    """Return Excel row numbers that belong to the 2026 reporting period."""
    source_last_row = last_used_row(worksheet)
    year_column = unique_header_column(headers, YEAR_HEADER, worksheet)
    years = column_values(worksheet, year_column, source_last_row)
    return [row_number for row_number, value in enumerate(years, start=2) if is_reporting_year(value)]


def values_for_rows(worksheet, column_number, rows):
    """Read one source column and keep only the requested Excel row numbers."""
    if not rows:
        return []
    values = column_values(worksheet, column_number, max(rows))
    return [values[row_number - 2] for row_number in rows]


def paste_column_from_staging(staging_sheet, worksheet, start_row, column_number, values):
    """Stage values on a blank sheet, then paste them into one master column."""
    if not values:
        return
    staging_range = staging_sheet.Range(
        staging_sheet.Cells(1, 1),
        staging_sheet.Cells(len(values), 1),
    )
    staging_range.Value2 = tuple((value,) for value in values)
    staging_range.Copy()
    worksheet.Range(
        worksheet.Cells(start_row, column_number),
        worksheet.Cells(start_row + len(values) - 1, column_number),
    ).PasteSpecial(Paste=XL_PASTE_VALUES)


def numeric_total(values):
    """Sum values while treating blank Excel cells as zero."""
    total = 0.0
    for value in values:
        if value not in (None, ""):
            total += float(value)
    return total


def build_column_mappings(master_headers, bsny_headers, sancap_headers):
    """Map each non-helper master column to the corresponding source-header occurrence."""
    master_to_source = {target: source for source, target in SOURCE_TO_MASTER.items()}
    source_occurrences = {}
    mappings = []

    for master_header, master_columns in master_headers.items():
        if master_header in HELPER_HEADERS:
            continue
        source_header = master_to_source.get(master_header, master_header)
        for master_column in master_columns:
            occurrence = source_occurrences.get(source_header, 0)
            source_occurrences[source_header] = occurrence + 1
            for source_name, source_headers in (("BSNY", bsny_headers), ("SanCap", sancap_headers)):
                available_columns = source_headers.get(source_header, [])
                if len(available_columns) <= occurrence:
                    raise ValueError(
                        f"{source_name} is missing occurrence {occurrence + 1} of source header "
                        f"'{source_header}' required for master column {master_column}."
                    )
            mappings.append((
                master_column,
                bsny_headers[source_header][occurrence],
                sancap_headers[source_header][occurrence],
            ))

    return mappings


def next_available_output_path(output_file):
    """Avoid overwriting a prior refreshed workbook, including one open in Excel."""
    candidate = output_file
    suffix_number = 1
    while candidate.exists():
        candidate = output_file.with_name(
            f"{output_file.stem}_{suffix_number}{output_file.suffix}"
        )
        suffix_number += 1
    return candidate


def resize_containing_table(worksheet, column_number, final_row):
    """Resize the table containing a target column before bulk data is written."""
    for table_index in range(1, worksheet.ListObjects.Count + 1):
        table = worksheet.ListObjects(table_index)
        first_column = table.Range.Column
        last_column = first_column + table.Range.Columns.Count - 1
        if first_column <= column_number <= last_column:
            table.Resize(worksheet.Range(
                worksheet.Cells(table.Range.Row, first_column),
                worksheet.Cells(final_row, last_column),
            ))
            return table.Name
    return None


excel = None
master_workbook = None
bsny_workbook = None
sancap_workbook = None
staging_sheet = None
refresh_succeeded = False
auto_fill_formulas_in_lists = None

try:
    if not OUTPUT_MASTER_FILE.parent.exists():
        raise RuntimeError("Run Step 10A before Step 10B.")

    # Create a new output copy before Excel opens any workbook.
    base_output_file = OUTPUT_DIR / f"{MASTER_FILE.stem}_REFRESHED{MASTER_FILE.suffix}"
    OUTPUT_MASTER_FILE = next_available_output_path(base_output_file)
    shutil.copy2(MASTER_FILE, OUTPUT_MASTER_FILE)

    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    excel.ScreenUpdating = False
    excel.EnableEvents = False

    master_workbook = excel.Workbooks.Open(str(OUTPUT_MASTER_FILE.resolve()))
    bsny_workbook = excel.Workbooks.Open(str(BSNY_CONCUR_FILE.resolve()), ReadOnly=True)
    sancap_workbook = excel.Workbooks.Open(str(SANCAP_CONCUR_FILE.resolve()), ReadOnly=True)
    excel.Calculation = XL_CALCULATION_MANUAL
    auto_fill_formulas_in_lists = excel.AutoCorrect.AutoFillFormulasInLists
    excel.AutoCorrect.AutoFillFormulasInLists = False

    master_sheet = master_workbook.Worksheets(MASTER_SHEET_NAME)
    bsny_sheet = bsny_workbook.Worksheets(BSNY_SHEET_NAME)
    sancap_sheet = sancap_workbook.Worksheets(1)
    staging_sheet = master_workbook.Worksheets.Add(After=master_workbook.Worksheets(master_workbook.Worksheets.Count))
    staging_sheet.Visible = XL_SHEET_VERY_HIDDEN

    master_headers = read_headers(master_sheet)
    bsny_headers = read_headers(bsny_sheet)
    sancap_headers = read_headers(sancap_sheet)

    # Helper, mapped, year, and expense columns are workbook structure and must be unique.
    helper_columns = {
        header: unique_header_column(master_headers, header, master_sheet)
        for header in HELPER_HEADERS
    }
    for target_header in SOURCE_TO_MASTER.values():
        unique_header_column(master_headers, target_header, master_sheet)
    master_year_column = unique_header_column(master_headers, YEAR_HEADER, master_sheet)
    master_expense_column = unique_header_column(master_headers, EXPENSE_HEADER, master_sheet)

    # Direct duplicate headers, such as Employee ID, are matched by first/second/etc. occurrence.
    column_mappings = build_column_mappings(master_headers, bsny_headers, sancap_headers)
    mapped_master_columns = [master_column for master_column, _, _ in column_mappings]
    if len(mapped_master_columns) != len(set(mapped_master_columns)):
        raise AssertionError("More than one source mapping targets the same master column.")

    bsny_rows = source_rows_for_year(bsny_sheet, bsny_headers)
    sancap_rows = source_rows_for_year(sancap_sheet, sancap_headers)
    if not bsny_rows or not sancap_rows:
        raise ValueError("No 2026 data found in one or both source workbooks. No data was cleared.")

    first_data_row = 2
    final_data_row = len(bsny_rows) + len(sancap_rows) + 1
    current_last_row = last_used_row(master_sheet)
    helper_formulas = {
        header: master_sheet.Cells(first_data_row, helper_columns[header]).FormulaR1C1
        for header in HELPER_HEADERS
    }
    missing_formulas = [header for header, formula in helper_formulas.items() if not str(formula).startswith("=")]
    if missing_formulas:
        raise ValueError("The formula template is missing in row 2 for: " + ", ".join(missing_formulas))

    resized_table = resize_containing_table(master_sheet, master_expense_column, final_data_row)
    if resized_table is not None:
        print(f"Resized Excel table: {resized_table}")

    print("STEP 10B: Refreshing Concur Report through Excel COM")
    print(f"BSNY 2026 rows:   {len(bsny_rows):,}")
    print(f"SanCap 2026 rows: {len(sancap_rows):,}")

    # Clear only source-data columns. Helper columns and their formulas are preserved.
    for master_column, _, _ in column_mappings:
        column_number = master_column
        master_sheet.Range(
            master_sheet.Cells(first_data_row, column_number),
            master_sheet.Cells(current_last_row, column_number),
        ).ClearContents()

    # Copy one target column at a time: first BSNY values, then SanCap values.
    expected_expense_values = None
    for master_column, bsny_column, sancap_column in column_mappings:
        bsny_values = values_for_rows(bsny_sheet, bsny_column, bsny_rows)
        sancap_values = values_for_rows(sancap_sheet, sancap_column, sancap_rows)
        combined_values = bsny_values + sancap_values
        paste_column_from_staging(staging_sheet, master_sheet, first_data_row, master_column, combined_values)
        if master_column == master_expense_column:
            expected_expense_values = combined_values
        if expected_expense_values is not None:
            current_expense_values = column_values(master_sheet, master_expense_column, final_data_row)
            if round(numeric_total(current_expense_values), 2) != round(numeric_total(expected_expense_values), 2):
                differences = [
                    (index + first_data_row, expected, actual)
                    for index, (expected, actual) in enumerate(zip(expected_expense_values, current_expense_values))
                    if expected != actual
                ]
                raise AssertionError(
                    f"Writing master column {master_column} "
                    f"('{master_sheet.Cells(1, master_column).Value2}') changed the expense amount values."
                    f" Excel formula in its first data cell: "
                    f"{master_sheet.Cells(first_data_row, master_column).Formula!r}"
                    f" Expected first values: {expected_expense_values[:5]!r}; "
                    f"written first values: {current_expense_values[:5]!r}."
                    f" First differing rows: {differences[:5]!r}"
                )

    written_expense_values = column_values(master_sheet, master_expense_column, final_data_row)
    if round(numeric_total(written_expense_values), 2) != round(numeric_total(expected_expense_values), 2):
        raise AssertionError("The expense column changed during source-data writes, before formula calculation.")

    # Remove the source Text format before filling relative helper formulas.
    for helper_header, formula in helper_formulas.items():
        helper_column = helper_columns[helper_header]
        formula_range = master_sheet.Range(
            master_sheet.Cells(first_data_row, helper_column),
            master_sheet.Cells(final_data_row, helper_column),
        )
        formula_range.Clear()
        formula_range.FormulaR1C1 = formula

    excel.Calculation = XL_CALCULATION_AUTOMATIC
    excel.CalculateFullRebuild()

    # Validate the saved-in-memory result before closing Excel.
    expected_rows = len(bsny_rows) + len(sancap_rows)
    actual_rows = final_data_row - first_data_row + 1
    if actual_rows != expected_rows:
        raise AssertionError(f"Expected {expected_rows:,} data rows but wrote {actual_rows:,}.")

    for helper_header in HELPER_HEADERS:
        helper_cell = master_sheet.Cells(final_data_row, helper_columns[helper_header])
        if not helper_cell.HasFormula:
            raise AssertionError(f"Helper formula was not filled to the last row: {helper_header}")

    year_column = master_year_column
    if not is_reporting_year(master_sheet.Cells(first_data_row, year_column).Value2):
        raise AssertionError("The first output row is not a 2026 BSNY record.")
    sancap_start_row = first_data_row + len(bsny_rows)
    if not is_reporting_year(master_sheet.Cells(sancap_start_row, year_column).Value2):
        raise AssertionError("The first SanCap output row is not a 2026 record.")

    expected_bsny_total = numeric_total(
        values_for_rows(bsny_sheet, unique_header_column(bsny_headers, EXPENSE_HEADER, bsny_sheet), bsny_rows)
    )
    expected_sancap_total = numeric_total(
        values_for_rows(sancap_sheet, unique_header_column(sancap_headers, EXPENSE_HEADER, sancap_sheet), sancap_rows)
    )
    expected_expense_total = expected_bsny_total + expected_sancap_total
    output_expense_values = column_values(master_sheet, master_expense_column, final_data_row)
    actual_bsny_total = numeric_total(output_expense_values[:len(bsny_rows)])
    actual_sancap_total = numeric_total(output_expense_values[len(bsny_rows):])
    actual_expense_total = actual_bsny_total + actual_sancap_total
    if round(actual_expense_total, 2) != round(expected_expense_total, 2):
        raise AssertionError(
            "Expense total mismatch. "
            f"BSNY expected/found: {expected_bsny_total:,.2f}/{actual_bsny_total:,.2f}; "
            f"SanCap expected/found: {expected_sancap_total:,.2f}/{actual_sancap_total:,.2f}."
        )

    staging_sheet.Visible = True
    staging_sheet.Delete()
    staging_sheet = None
    master_workbook.Save()
    refresh_succeeded = True
    print(f"Data rows written: {expected_rows:,} (BSNY first, then SanCap)")
    print(f"Expense total: ${actual_expense_total:,.2f}")
    print(f"Saved refreshed file: {OUTPUT_MASTER_FILE}")
    

finally:
    if bsny_workbook is not None:
        bsny_workbook.Close(SaveChanges=False)
    if sancap_workbook is not None:
        sancap_workbook.Close(SaveChanges=False)
    if staging_sheet is not None:
        try:
            staging_sheet.Visible = True
            staging_sheet.Delete()
        except Exception:
            pass
    if master_workbook is not None:
        master_workbook.Close(SaveChanges=refresh_succeeded)
    if excel is not None:
        if auto_fill_formulas_in_lists is not None:
            excel.AutoCorrect.AutoFillFormulasInLists = auto_fill_formulas_in_lists
        excel.Quit()
    excel = None
    master_workbook = None
    bsny_workbook = None
    sancap_workbook = None

STEP 10B: Refreshing Concur Report through Excel COM
BSNY 2026 rows:   15,377
SanCap 2026 rows: 33,177
Data rows written: 48,554 (BSNY first, then SanCap)
Expense total: $10,806,142.68
Saved refreshed file: outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED_REFRESHED_1.xlsx


In [5]:
# STEP 10C: Publish the validated refresh under the stable output filename
# Run this only after Step 10B completes successfully.

PUBLISHED_OUTPUT_FILE = OUTPUT_DIR / f"{MASTER_FILE.stem}_paste_concur{MASTER_FILE.suffix}"

if not OUTPUT_MASTER_FILE.exists():
    raise FileNotFoundError(
        "The validated Step 10B output was not found. Run Step 10B before publishing."
    )

shutil.copy2(OUTPUT_MASTER_FILE, PUBLISHED_OUTPUT_FILE)
OUTPUT_MASTER_FILE = PUBLISHED_OUTPUT_FILE

print(f"Published refreshed file: {OUTPUT_MASTER_FILE}")
print("An existing _paste_concur.xlsx file was overwritten.")

Published refreshed file: outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED_paste_concur.xlsx
An existing _paste_concur.xlsx file was overwritten.


In [6]:
# STEP 10D: Rebuild helper cells as calculated Excel formulas
# Run after Step 10C. This changes only the five helper columns.

excel = None
output_workbook = None
repair_succeeded = False

try:
    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    excel.ScreenUpdating = False

    output_workbook = excel.Workbooks.Open(str(PUBLISHED_OUTPUT_FILE.resolve()))
    output_sheet = output_workbook.Worksheets(MASTER_SHEET_NAME)
    output_headers = read_headers(output_sheet)
    output_last_row = last_used_row(output_sheet)

    for helper_header in HELPER_HEADERS:
        helper_column = unique_header_column(output_headers, helper_header, output_sheet)
        helper_range = output_sheet.Range(
            output_sheet.Cells(2, helper_column),
            output_sheet.Cells(output_last_row, helper_column),
        )
        formula_template = output_sheet.Cells(2, helper_column).FormulaR1C1
        if not str(formula_template).startswith("="):
            raise ValueError(f"Missing formula template in '{helper_header}'.")

        # The source helper cells are Text-formatted, which makes Excel display
        # reassigned formulas literally. Clear removes that format before filling.
        helper_range.Clear()
        helper_range.FormulaR1C1 = formula_template

        first_cell = output_sheet.Cells(2, helper_column)
        if not first_cell.HasFormula:
            raise AssertionError(f"'{helper_header}' was not converted into an Excel formula.")

    excel.CalculateFullRebuild()
    output_workbook.Save()
    repair_succeeded = True
    print(f"Helper formulas recalculated in: {PUBLISHED_OUTPUT_FILE}")
finally:
    if output_workbook is not None:
        output_workbook.Close(SaveChanges=repair_succeeded)
    if excel is not None:
        excel.Quit()

Helper formulas recalculated in: outputs\Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED_paste_concur.xlsx
